In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

gold_country_daily = spark.table(
    "workspace.gold.country_ingestion_daily"
)

display(gold_country_daily.limit(10))

In [0]:
country_window = (
    Window
    .partitionBy("CountryCode")
    .orderBy("AddedDate")
)

rolling_window = (
    country_window
    .rowsBetween(-6, -1)
)

country_features = (
    gold_country_daily

    .withColumn(
        "PreviousEventCount",
        F.lag("EventCount", 1).over(country_window)
    )

    .withColumn(
        "EventCountChange",
        F.col("EventCount") -
        F.col("PreviousEventCount")
    )

    .withColumn(
        "EventCountRatio",
        F.when(
            F.col("PreviousEventCount") > 0,
            F.col("EventCount") /
            F.col("PreviousEventCount")
        )
    )

    .withColumn(
        "RollingMean7",
        F.avg("EventCount").over(rolling_window)
    )

    .withColumn(
        "RollingStd7",
        F.stddev("EventCount").over(rolling_window)
    )

    .withColumn(
        "DeviationFromRollingMean",
        F.when(
            F.col("RollingMean7").isNotNull(),
            F.col("EventCount") -
            F.col("RollingMean7")
        )
    )

    .withColumn(
        "RollingZScore",
        F.when(
            F.col("RollingStd7") > 0,
            (
                F.col("EventCount") -
                F.col("RollingMean7")
            ) / F.col("RollingStd7")
        )
    )
)

In [0]:
(
    country_features.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.country_anomaly_features"
    )
)

In [0]:
print(
    "Feature rows:",
    spark.table(
        "workspace.gold.country_anomaly_features"
    ).count()
)

In [0]:
display(
    spark.sql("""
        SELECT
            AddedDate,
            CountryCode,
            EventCount,
            PreviousEventCount,
            EventCountChange,
            EventCountRatio,
            RollingMean7,
            RollingStd7,
            RollingZScore
        FROM workspace.gold.country_anomaly_features
        WHERE RollingZScore IS NOT NULL
        ORDER BY ABS(RollingZScore) DESC
        LIMIT 30
    """)
)

In [0]:
display(
    spark.sql("""
        SELECT
            AddedDate,
            CountryCode,
            EventCount,
            PreviousEventCount,
            EventCountChange,
            EventCountRatio,
            RollingMean7,
            RollingStd7,
            RollingZScore
        FROM workspace.gold.country_anomaly_features
        WHERE RollingZScore IS NOT NULL
        ORDER BY ABS(RollingZScore) DESC
        LIMIT 30
    """)
)

In [0]:
display(
    spark.sql("""
        SELECT
            CountryCode,
            COUNT(*) AS days_available,
            MIN(AddedDate) AS first_date,
            MAX(AddedDate) AS last_date
        FROM workspace.gold.country_anomaly_features
        GROUP BY CountryCode
        ORDER BY days_available DESC
        LIMIT 20
    """)
)

In [0]:
display(
    spark.sql("""
        SELECT
            AddedDate,
            COUNT(*) AS total_events,
            COUNT(DISTINCT CountryCode) AS countries
        FROM workspace.gold.country_ingestion_daily
        GROUP BY AddedDate
        ORDER BY AddedDate
    """)
)

In [0]:
display(
    spark.sql("""
        SELECT
            CountryCode,
            COUNT(*) AS days_available,
            MIN(AddedDate) AS first_date,
            MAX(AddedDate) AS last_date
        FROM workspace.gold.country_anomaly_features
        GROUP BY CountryCode
        ORDER BY days_available DESC
        LIMIT 20
    """)
)

In [0]:
from pyspark.sql import functions as F

feature_ready = (
    spark.table("workspace.gold.country_anomaly_features")
    .filter(F.col("RollingMean7").isNotNull())
    .filter(F.col("RollingStd7").isNotNull())
)

In [0]:
print("Feature rows with 7-day history:", feature_ready.count())

In [0]:
print("Feature rows with 7-day history:", feature_ready.count())

In [0]:
print("Usable ML rows:", feature_ready.count())

In [0]:
(
    feature_ready.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.country_anomaly_features_ready"
    )
)

In [0]:
display(
    spark.sql("""
        SELECT
            AddedDate,
            CountryCode,
            EventCount,
            PreviousEventCount,
            EventCountChange,
            EventCountRatio,
            RollingMean7,
            RollingStd7,
            RollingZScore,
            TotalMentions,
            TotalSources,
            TotalArticles,
            AvgGoldsteinScale,
            AvgTone
        FROM workspace.gold.country_anomaly_features_ready
        ORDER BY ABS(RollingZScore) DESC
        LIMIT 30
    """)
)